In [ ]:
import random
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names

# 💡 데이터셋 정보: eliceai/korean-webtext-edu
# 📄 제목: 한국 웹 텍스트 교육 자료 (Korean Webtext Educational Corpus)
# 📖 의미: 방대한 한국 웹에서 '교육적 가치'가 높다고 평가된 고품질의 학습 자료 텍스트 모음입니다.
# 🎯 실습 목표: 텍스트의 길이(정보량)와 데이터셋이 부여한 '교육적 점수' 사이의 관계를 탐색해보고, 데이터의 분포를 시각적으로 이해하는 방법을 배웁니다!

DATASET_NAME = "eliceai/korean-webtext-edu"
SAMPLE_COUNT = 15 # 코드를 실행하며 천천히 탐색해 볼 15개의 샘플만 가져와요!

print("="*70)
print(f"✨ AI 튜터와 함께하는 {DATASET_NAME} 데이터 탐험 시작!")
print("="*70)

# 1. 사용 가능한 Config 확인 (필수 단계예요!)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 만약 여러 config가 있다면, 첫 번째 기본 설정을 사용합니다.
    if configs:
        selected_config = configs[0]
    else:
        selected_config = None

except Exception as e:
    print(f"⚠️ Config 목록 로드 중 오류가 발생했으나, 기본 설정으로 진행합니다. ({e})")
    selected_config = None

# 2. 스트리밍 모드로 데이터셋 로드 (메모리 걱정 NO!)
# streaming=True를 사용하여 데이터셋 전체를 한 번에 메모리에 올리지 않아요.
try:
    print(f"\n🚀 '{DATASET_NAME}' 데이터셋을 스트리밍 방식으로 로드합니다...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
except Exception as e:
    print(f"❌ 데이터셋 로드에 실패했습니다. 인터넷 연결이나 데이터셋 이름 확인이 필요합니다. ({e})")
    exit()

# 3. 샘플링 전략 (스트리밍 환경 최적화)
# streaming 모드에서는 len()이나 random.sample()을 쓸 수 없기 때문에,
# .take()를 이용해 원하는 개수만큼 샘플을 가져와야 해요.
print(f"🔎 상위 {SAMPLE_COUNT}개의 샘플을 가져와 분석을 시작할게요...")
# 데이터를 반복 가능한(iterable) 객체로 만들고, 첫 N개만 가져옵니다.
sample_iterator = iter(dataset.take(SAMPLE_COUNT)) 

# 4. 첫 번째 샘플 구조 확인 (워밍업 단계!)
print("\n🌟 [Step 1: 데이터 구조 살펴보기]")
print("--- 첫 번째 샘플을 한번 꺼내와 구조를 살펴봐요! (Don't forget the first sample!)")

try:
    first_sample = next(sample_iterator)
    print(f"  📝 샘플 데이터 타입: {type(first_sample)}")
    print(f"  📚 텍스트 (text): {first_sample['text'][:80]}...")
    print(f"  ✨ 점수 (score): {first_sample['score']:.2f}")
    print("-------------------------------------------------------")
except StopIteration:
    print("⚠️ 가져올 샘플이 없습니다. 데이터셋을 확인해주세요.")
    exit()


# 5. 모든 샘플을 리스트로 변환 및 전처리 (분석에 사용하기 위함!)
# 이제 남은 반복자를 끝까지 순회하며, 분석에 필요한 데이터만 모아봅시다.
print(f"\n🧠 [Step 2: {SAMPLE_COUNT}개 샘플 데이터를 수집하고 전처리합니다]")
sample_list = []
for i in range(SAMPLE_COUNT):
    try:
        sample = next(sample_iterator)
        # 텍스트와 점수만 필요하므로 리스트에 저장합니다.
        sample_list.append({'text': sample['text'], 'score': sample['score']})
    except StopIteration:
        break
    
print(f"   ✅ 총 {len(sample_list)}개의 데이터를 준비했습니다. 최고!")


# 6. 창의적 실습: 텍스트 길이 vs. 교육적 점수 관계 시각화
print("\n📈 [Step 3: 텍스트 길이와 점수의 관계를 시각화해볼까요?]")
print("   (힌트: 텍스트가 길다고 무조건 점수가 높은 건 아닐 수 있어요! 😉)")

# 시각화를 위한 데이터 준비
text_lengths = []
scores = []

for sample in sample_list:
    # 텍스트의 단어 수 또는 글자 수를 측정합니다. (단순히 글자 수로 측정해볼게요!)
    length = len(sample['text'])
    text_lengths.append(length)
    scores.append(sample['score'])

# Matplotlib를 사용해 점수와 길이를 Scatter Plot으로 그려봅니다.
plt.figure(figsize=(10, 6))
plt.scatter(text_lengths, scores, alpha=0.6) # alpha를 주면 데이터 점들이 겹쳐도 예뻐요!

plt.title('Relationship between Text Length and Educational Score', fontsize=14)
plt.xlabel('Text Length (Number of Characters)', fontsize=12)
plt.ylabel('Educational Score (Quality Metric)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# 🎓 튜터의 한 마디: 이 그래프를 보면, 긴 글이 항상 점수가 높다는 확실한 규칙은 보이지 않네요. 
# 하지만 점수가 높은 샘플들이 어느 정도 길이 범위에 몰려있는지 패턴을 찾을 수 있을 거예요!
plt.tight_layout()
plt.show()

print("\n="*70)
print("✨ 🎉 실습 완료! 축하드립니다! 🎉 ✨")
print("교육 데이터셋의 핵심 특성을 성공적으로 파악하셨어요!")
print("데이터 분석은 이렇게 작은 '탐색'부터 시작됩니다. 다음 단계로 넘어가 볼까요? 😉")